# ENTSO-E — cross-border physical flow ingestion
Ingests hourly physical flows for six key interconnectors across five countries.  
Data is stored in the `cross_border_flows` table as raw directional MW values.

## Interconnectors covered
| From  | To    | Notes                        |
|-------|-------|------------------------------|
| FR    | DE_LU | Major nuclear export route   |
| DE_LU | FR    |                              |
| FR    | ES    |                              |
| ES    | FR    |                              |
| FR    | GB    | IFA interconnector           |
| GB    | FR    |                              |
| DE_LU | IT    |                              |
| IT    | DE_LU |                              |
| ES    | IT    |                              |
| IT    | ES    |                              |

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
import time
import psycopg2
import pandas as pd
from entsoe import EntsoePandasClient
from datetime import timezone

load_dotenv(find_dotenv(), override=True)

ENTSOE_TOKEN = os.environ["ENTSOE_TOKEN"]
DB_URL       = os.environ["DATABASE_URL"]

conn = psycopg2.connect(DB_URL)

def query(sql, params=None):
    return pd.read_sql(sql, conn, params=params)

print("DB connected ✓")

DB connected ✓


In [2]:
# ── Config ────────────────────────────────────────────────────────

# ENTSO-E EIC codes for cross-border physical flow queries
# Note: GB uses 10YGB----------A (not the BZN code 10Y1001A1001A59C
# which is generation/load only — GB post-Brexit split)
FLOW_CODES = {
    "DE_LU": "10Y1001A1001A82H",
    "FR":    "10YFR-RTE------C",
    "ES":    "10YES-REE------0",
    "IT":    "10YIT-GRTN-----B",
    "GB":    "10YGB----------A",
}

# Directional pairs — both directions stored separately
# Raw directional MW values; net flow derived at query time
FLOW_PAIRS = [
    ("FR",    "DE_LU"),
    ("DE_LU", "FR"),
    ("FR",    "ES"),
    ("ES",    "FR"),
    ("FR",    "GB"),    # IFA interconnector
    ("GB",    "FR"),
    ("FR",    "IT"),    # direct Alpine interconnectors
    ("IT",    "FR"),
]

print(f"{len(FLOW_PAIRS)} directional pairs configured")

8 directional pairs configured


In [3]:
# ── DB helpers ────────────────────────────────────────────────────

def log_note(conn, country, note_type, description):
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO data_notes (country, note_type, description) VALUES (%s, %s, %s)",
            (country, note_type, description),
        )
    conn.commit()


def insert_flows(conn, from_country: str, to_country: str, df: pd.DataFrame):
    """Insert cross-border flow data into cross_border_flows table.
    
    ENTSO-E returns 15-min or hourly data depending on the country pair.
    Resampled to hourly via mean() to align with generation table.
    NULL = reported gap; 0.0 = genuine zero flow.
    """

    # query_crossborder_flows returns a Series, not a DataFrame
    if isinstance(df, pd.Series):
        df = df.to_frame(name='mw')
    else:
        df.columns = ['mw']

    # Resample to hourly — align with generation table granularity
    df = df.resample('h').mean()

    rows = []
    for ts, row in df.iterrows():
        ts_utc = ts.astimezone(timezone.utc).isoformat()
        mw_val = row['mw']
        rows.append({
            "from_country":  from_country,
            "to_country":    to_country,
            "timestamp_utc": ts_utc,
            "mw":            None if pd.isna(mw_val) else float(mw_val),
        })

    with conn.cursor() as cur:
        cur.executemany(
            """
            INSERT INTO cross_border_flows (from_country, to_country, timestamp_utc, mw)
            VALUES (%(from_country)s, %(to_country)s, %(timestamp_utc)s, %(mw)s)
            ON CONFLICT (from_country, to_country, timestamp_utc) DO UPDATE
                SET mw = EXCLUDED.mw
            """,
            rows,
        )
    conn.commit()
    return len(rows)


def ingest_pair_year(client, conn, from_country: str, to_country: str, year: int):
    """Ingest one directional flow pair for a full year, month by month."""

    start = pd.Timestamp(f"{year}-01-01", tz="Europe/Brussels")
    end   = pd.Timestamp(f"{year+1}-01-01", tz="Europe/Brussels")
    total = 0

    for month_start in pd.date_range(start, end, freq="MS"):
        month_end = month_start + pd.offsets.MonthEnd(1) + pd.Timedelta(days=1)
        month_end = min(month_end, end)

        label = month_start.strftime("%Y-%m")
        pair  = f"{from_country}→{to_country}"
        print(f"    {pair} {label}...", end=" ", flush=True)

        try:
            df = client.query_crossborder_flows(
                country_code_from=FLOW_CODES[from_country],
                country_code_to=FLOW_CODES[to_country],
                start=month_start,
                end=month_end,
            )
            n = insert_flows(conn, from_country, to_country, df)
            total += n
            print(f"{n} rows")

        except Exception as e:
            print(f"ERROR — {type(e).__name__}: {e}")
            log_note(conn, from_country, "gap",
                     f"crossborder {pair} {label}: {str(e)[:200]}")

        time.sleep(2)  # polite delay between requests

    return total

In [4]:
# ── Main — ingest all pairs for 2024, 2025, 2026 ─────────────────

client = EntsoePandasClient(api_key=ENTSOE_TOKEN)

for from_country, to_country in FLOW_PAIRS:
    for year in [2024, 2025, 2026]:
        print(f"\nIngesting {from_country}→{to_country} {year}...")
        total = ingest_pair_year(client, conn, from_country, to_country, year)
        print(f"→ {total} total rows for {from_country}→{to_country} {year}")


Ingesting FR→DE_LU 2024...
    FR→DE_LU 2024-01... 744 rows
    FR→DE_LU 2024-02... 696 rows
    FR→DE_LU 2024-03... 744 rows
    FR→DE_LU 2024-04... 720 rows
    FR→DE_LU 2024-05... 744 rows
    FR→DE_LU 2024-06... 720 rows
    FR→DE_LU 2024-07... 744 rows
    FR→DE_LU 2024-08... 744 rows
    FR→DE_LU 2024-09... 720 rows
    FR→DE_LU 2024-10... 745 rows
    FR→DE_LU 2024-11... 720 rows
    FR→DE_LU 2024-12... 744 rows
    FR→DE_LU 2025-01... ERROR — NoMatchingDataError: 
→ 8785 total rows for FR→DE_LU 2024

Ingesting FR→DE_LU 2025...
    FR→DE_LU 2025-01... 744 rows
    FR→DE_LU 2025-02... 672 rows
    FR→DE_LU 2025-03... 743 rows
    FR→DE_LU 2025-04... 720 rows
    FR→DE_LU 2025-05... 744 rows
    FR→DE_LU 2025-06... 720 rows
    FR→DE_LU 2025-07... 744 rows
    FR→DE_LU 2025-08... 744 rows
    FR→DE_LU 2025-09... 720 rows
    FR→DE_LU 2025-10... 745 rows
    FR→DE_LU 2025-11... 720 rows
    FR→DE_LU 2025-12... 744 rows
    FR→DE_LU 2026-01... ERROR — NoMatchingDataError: 
→ 8760 t

In [5]:
# ── Quick sanity check ────────────────────────────────────────────
# Row counts per directional pair — confirm all pairs ingested

summary = query("""
    SELECT
        from_country,
        to_country,
        COUNT(*)                                    AS total_rows,
        COUNT(mw)                                   AS non_null_rows,
        ROUND(100.0 * (COUNT(*) - COUNT(mw))
              / NULLIF(COUNT(*), 0), 1)             AS null_pct,
        MIN(timestamp_utc)::date                    AS first_date,
        MAX(timestamp_utc)::date                    AS last_date
    FROM cross_border_flows
    GROUP BY from_country, to_country
    ORDER BY from_country, to_country
""")

print(summary.to_string())

  from_country to_country  total_rows  non_null_rows  null_pct  first_date   last_date
0        DE_LU         FR       21419          21419       0.0  2023-12-31  2026-06-11
1           ES         FR       21420          21385       0.2  2023-12-31  2026-06-11
2           FR      DE_LU       21419          21419       0.0  2023-12-31  2026-06-11
3           FR         ES       21420          21420       0.0  2023-12-31  2026-06-11
4           FR         GB       21419          21417       0.0  2023-12-31  2026-06-11
5           FR         IT       21419          21419       0.0  2023-12-31  2026-06-11
6           GB         FR       21419          21417       0.0  2023-12-31  2026-06-11
7           IT         FR       21419          21419       0.0  2023-12-31  2026-06-11


/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_67710/2425857931.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


In [6]:
# ── Net flow preview ──────────────────────────────────────────────
# Derive net flow per pair (positive = net export from 'from_country')
# This is a preview only — full analysis in data_completeness_audit.ipynb Section G

net_flows = query("""
    WITH forward AS (
        SELECT from_country, to_country, timestamp_utc, mw AS mw_forward
        FROM cross_border_flows
    ),
    reverse AS (
        SELECT to_country AS from_country, from_country AS to_country,
               timestamp_utc, mw AS mw_reverse
        FROM cross_border_flows
    )
    SELECT
        f.from_country,
        f.to_country,
        ROUND(AVG(COALESCE(f.mw_forward, 0) - COALESCE(r.mw_reverse, 0)), 1) AS avg_net_mw,
        ROUND(MIN(COALESCE(f.mw_forward, 0) - COALESCE(r.mw_reverse, 0)), 1) AS min_net_mw,
        ROUND(MAX(COALESCE(f.mw_forward, 0) - COALESCE(r.mw_reverse, 0)), 1) AS max_net_mw
    FROM forward f
    JOIN reverse r
        ON f.from_country = r.from_country
        AND f.to_country  = r.to_country
        AND f.timestamp_utc = r.timestamp_utc
    GROUP BY f.from_country, f.to_country
    ORDER BY f.from_country, f.to_country
""")

print("Net flow summary (positive = net export from from_country):")
print(net_flows.to_string())

/var/folders/7k/t37lnf710tz5jtzx8d0jzqbc0000gq/T/ipykernel_67710/2425857931.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


Net flow summary (positive = net export from from_country):
  from_country to_country  avg_net_mw  min_net_mw  max_net_mw
0        DE_LU         FR     -2256.1     -4704.5      4213.0
1           ES         FR      -120.9     -3686.1      3656.0
2           FR      DE_LU      2256.1     -4213.0      4704.5
3           FR         ES       120.9     -3656.0      3686.1
4           FR         GB      2491.3     -3581.5      4089.0
5           FR         IT      2414.4     -2119.0      4963.4
6           GB         FR     -2491.3     -4089.0      3581.5
7           IT         FR     -2414.4     -4963.4      2119.0
